In [1]:
!pip install transformers torch sentencepiece -q

In [2]:
# ============================================================
#   🏥 PLUM OPD CLAIM ADJUDICATION TOOL
#   ✅ Interactive Frontend — runs in Google Colab
#   Built with ipywidgets — No HTML/React needed!
#   ✅ Bug Fixed: Teeth whitening bill amount corrected
# ============================================================

In [3]:
# ================================
# 1. INSTALL & IMPORTS
# Run this first:
# !pip install transformers torch sentencepiece ipywidgets -q
# ================================
import json
import re
import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from datetime import datetime, timedelta

print("✅ Libraries imported!")

✅ Libraries imported!


In [4]:
# ================================
# 2. LOAD FREE AI MODEL
# ================================
print("⏳ Loading free AI model (first time ~60 sec)...")

MODEL_NAME = "google/flan-t5-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
ai_model   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
ai_model.eval()

def ask_ai(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = ai_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=2)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("🤖 Free AI model ready! (google/flan-t5-base)")
test_out = ask_ai("What is the capital of India?")
print(f"   Self-test: {test_out}")

⏳ Loading free AI model (first time ~60 sec)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

🤖 Free AI model ready! (google/flan-t5-base)
   Self-test: mumbai


In [5]:
# ================================
# 3. LOAD FILES
# ================================
with open("policy_terms.json") as f:
    policy = json.load(f)

with open("test_cases.json") as f:
    test_cases = json.load(f)

print("✅ policy_terms.json loaded")
print(f"   💰 Per-claim limit  : ₹{policy['coverage_details']['per_claim_limit']:,}")
print(f"   📅 Annual limit     : ₹{policy['coverage_details']['annual_limit']:,}")
print(f"   🏥 Network hospitals: {len(policy['network_hospitals'])}")
print(f"\n✅ test_cases.json loaded")
print(f"   🧪 Total test cases : {len(test_cases['test_cases'])}")

✅ policy_terms.json loaded
   💰 Per-claim limit  : ₹5,000
   📅 Annual limit     : ₹50,000
   🏥 Network hospitals: 5

✅ test_cases.json loaded
   🧪 Total test cases : 10


In [6]:
# ================================
# 4. RULE ENGINE
# ================================
def adjudicate_claim(claim, policy):
    """
    Rules in priority order:
      1.  Missing / empty prescription
      2.  Fraud — multiple claims same day
      3.  Exclusion — weight loss / obesity
      4.  Waiting period — diabetes
      5.  Partial approval — cosmetic dental
      6.  MRI pre-authorization
      7.  Per-claim limit exceeded
      8.  Network hospital cashless
      9.  Alternative medicine
      10. Default approval with 10% copay
    """
    claim_amount = claim.get("claim_amount", 0)
    docs         = claim.get("documents", {})
    prescription = docs.get("prescription", {})
    bill         = docs.get("bill", {})
    diagnosis    = prescription.get("diagnosis", "").lower()
    procedures   = prescription.get("procedures", [])
    bill_str     = str(bill).lower()
    presc_str    = str(prescription).lower()

    # RULE 1: Missing / empty prescription
    if not prescription or len(prescription) == 0:
        return {
            "decision"         : "REJECTED",
            "rejection_reasons": ["MISSING_DOCUMENTS"],
            "notes"            : "Prescription from registered doctor is required",
            "confidence_score" : 1.0
        }

    # RULE 2: Fraud — multiple claims same day
    if claim.get("previous_claims_same_day", 0) >= 3:
        return {
            "decision"        : "MANUAL_REVIEW",
            "flags"           : ["Multiple claims same day", "Unusual pattern detected"],
            "notes"           : "Flagged for human review due to suspicious claim frequency",
            "confidence_score": 0.65
        }

    # RULE 3: Exclusion — weight loss / obesity
    if "obesity" in diagnosis or "weight loss" in diagnosis:
        return {
            "decision"         : "REJECTED",
            "rejection_reasons": ["SERVICE_NOT_COVERED"],
            "notes"            : "Weight loss treatments are excluded from coverage",
            "confidence_score" : 0.97
        }

    # RULE 4: Waiting period — diabetes
    if "diabetes" in diagnosis:
        eligible_note = ""
        join_date = claim.get("member_join_date")
        if join_date:
            try:
                jd            = datetime.strptime(join_date, "%Y-%m-%d")
                eligible_date = jd + timedelta(days=90)
                eligible_note = f" Eligible from {eligible_date.strftime('%Y-%m-%d')}"
            except:
                pass
        return {
            "decision"         : "REJECTED",
            "rejection_reasons": ["WAITING_PERIOD"],
            "notes"            : f"Diabetes has 90-day waiting period.{eligible_note}",
            "confidence_score" : 0.96
        }

    # RULE 5: Partial approval — cosmetic dental
    if "Teeth whitening" in procedures:
        cosmetic_amount = bill.get("teeth_whitening", 0)
        approved        = claim_amount - cosmetic_amount
        return {
            "decision"        : "PARTIAL",
            "approved_amount" : approved,
            "rejected_items"  : ["Teeth whitening - cosmetic procedure"],
            "notes"           : f"Root canal approved. Cosmetic amount ₹{cosmetic_amount:,} excluded.",
            "confidence_score": 0.92
        }

    # RULE 6: MRI — pre-authorization required
    if ("mri" in bill_str or "mri" in presc_str) and claim_amount > 10000:
        return {
            "decision"         : "REJECTED",
            "rejection_reasons": ["PRE_AUTH_MISSING"],
            "notes"            : "MRI requires pre-authorization for claims above ₹10,000",
            "confidence_score" : 0.94
        }

    # RULE 7: Per-claim limit exceeded
    per_claim_limit = policy["coverage_details"]["per_claim_limit"]
    if claim_amount > per_claim_limit:
        return {
            "decision"         : "REJECTED",
            "rejection_reasons": ["PER_CLAIM_EXCEEDED"],
            "notes"            : f"Claim amount exceeds per-claim limit of ₹{per_claim_limit:,}",
            "confidence_score" : 0.98
        }

    # RULE 8: Network hospital — cashless with 20% discount
    if claim.get("hospital") in policy["network_hospitals"]:
        discount_pct = policy["coverage_details"]["consultation_fees"]["network_discount"] / 100
        discount     = int(claim_amount * discount_pct)
        return {
            "decision"         : "APPROVED",
            "approved_amount"  : claim_amount - discount,
            "cashless_approved": True,
            "network_discount" : discount,
            "notes"            : f"Network hospital benefit. 20% discount of ₹{discount:,} applied.",
            "confidence_score" : 0.93
        }

    # RULE 9: Alternative medicine
    if any(t in presc_str for t in ["ayur", "homeo", "unani", "panchakarma"]):
        return {
            "decision"        : "APPROVED",
            "approved_amount" : claim_amount,
            "notes"           : "Alternative medicine covered under policy",
            "confidence_score": 0.89
        }

    # RULE 10: Default — approve with 10% copay
    copay_pct = policy["coverage_details"]["consultation_fees"]["copay_percentage"] / 100
    copay     = int(claim_amount * copay_pct)
    return {
        "decision"        : "APPROVED",
        "approved_amount" : claim_amount - copay,
        "deductions"      : {"copay": copay},
        "notes"           : f"Standard approval with 10% co-pay of ₹{copay:,}",
        "confidence_score": 0.95
    }

In [7]:
# ================================
# 5. AI EXPLANATION
# ================================
def get_ai_explanation(claim, result):
    decision = result.get("decision", "")
    prompt   = (
        f"Explain this insurance claim result in one simple sentence for the patient.\n"
        f"Patient: {claim.get('member_name')}. "
        f"Claimed: Rs {claim.get('claim_amount')}. "
        f"Decision: {decision}. "
        f"Reason: {result.get('notes', '')}\n"
        f"Explanation:"
    )
    try:
        return ask_ai(prompt, max_new_tokens=80)
    except:
        return result.get("notes", "")

In [8]:
# ================================
# 6. RESULT HTML CARD BUILDER
# ================================
def build_result_html(claim, result, ai_text):
    decision = result.get("decision", "")

    styles = {
        "APPROVED"     : ("#1b5e20", "#e8f5e9", "#43a047", "✅"),
        "REJECTED"     : ("#b71c1c", "#ffebee", "#e53935", "❌"),
        "PARTIAL"      : ("#e65100", "#fff3e0", "#fb8c00", "⚠️"),
        "MANUAL_REVIEW": ("#0d47a1", "#e3f2fd", "#1e88e5", "🔍")
    }
    text_color, bg_color, accent, emoji = styles.get(decision, ("#333", "#fafafa", "#999", "📋"))

    def row(label, value, bold=False):
        fw = "font-weight:600;" if bold else ""
        return (f"<tr>"
                f"<td style='padding:8px 16px;color:#666;width:160px;"
                f"border-bottom:1px solid #f0f0f0'>{label}</td>"
                f"<td style='padding:8px 16px;{fw}border-bottom:1px solid #f0f0f0'>{value}</td>"
                f"</tr>")

    rows = ""
    rows += row("👤 Member",    claim.get("member_name", "N/A"))
    rows += row("📅 Date",      claim.get("treatment_date", "N/A"))
    rows += row("🏥 Hospital",  claim.get("hospital") or "Non-network")
    rows += row("💵 Claimed",   f"₹{claim.get('claim_amount', 0):,}")
    rows += row("🏷️ Decision",
                f"<span style='color:{text_color};font-weight:700;font-size:15px'>"
                f"{emoji} {decision}</span>")
    rows += row("📊 Confidence", str(result.get("confidence_score", "N/A")))

    if result.get("approved_amount") is not None:
        rows += row("💰 Approved",
                    f"<span style='color:{text_color};font-weight:700;font-size:16px'>"
                    f"₹{result['approved_amount']:,}</span>", bold=True)
    if result.get("network_discount"):
        rows += row("🏥 Discount",
                    f"₹{result['network_discount']:,} (20% network benefit)")
    if result.get("deductions"):
        for k, v in result["deductions"].items():
            rows += row(f"✂️ Copay", f"₹{v:,} ({k})")
    if result.get("rejection_reasons"):
        rows += row("🚫 Reason",
                    f"<span style='color:#c62828'>"
                    f"{', '.join(result['rejection_reasons'])}</span>")
    if result.get("rejected_items"):
        rows += row("🚫 Excluded",
                    f"<span style='color:#e65100'>"
                    f"{', '.join(result['rejected_items'])}</span>")
    if result.get("flags"):
        rows += row("🚩 Flags",
                    f"<span style='color:#1565c0'>"
                    f"{', '.join(result['flags'])}</span>")
    if result.get("notes"):
        rows += row("📝 Notes", result["notes"])

    return f"""
    <div style="font-family:Arial,sans-serif;max-width:660px;margin:16px auto;
                border-radius:14px;overflow:hidden;
                box-shadow:0 6px 24px rgba(0,0,0,0.13)">

      <!-- Header bar -->
      <div style="background:{accent};padding:4px 0"></div>

      <!-- Title -->
      <div style="background:{text_color};padding:18px 22px;color:white">
        <div style="font-size:22px;font-weight:700">{emoji} {decision}</div>
        <div style="font-size:12px;opacity:0.8;margin-top:4px">
          Plum OPD Advantage &nbsp;·&nbsp;
          {datetime.now().strftime('%d %b %Y, %I:%M %p')}
        </div>
      </div>

      <!-- Details table -->
      <div style="background:{bg_color}">
        <table style="width:100%;border-collapse:collapse">{rows}</table>
      </div>

      <!-- AI explanation -->
      <div style="background:#fafafa;padding:14px 22px;border-top:2px solid {accent}">
        <div style="font-size:11px;color:#999;margin-bottom:6px;text-transform:uppercase;
                    letter-spacing:0.5px">🤖 AI Explanation</div>
        <div style="font-size:14px;color:#333;line-height:1.6">{ai_text}</div>
      </div>

    </div>
    """

In [9]:
# ================================
# 7. BUILD THE UI
# ================================

# ── Helper ───────────────────────────────────────────────────────────────
def lbl(text):
    return widgets.HTML(
        f"<div style='font-size:12px;font-weight:600;color:#444;"
        f"margin-top:10px;margin-bottom:2px'>{text}</div>"
    )

W = widgets.Layout(width="320px")

# ── Tab 1 widgets ────────────────────────────────────────────────────────
w_name        = widgets.Text(placeholder="e.g. Rajesh Kumar", layout=W)
w_member_id   = widgets.Text(placeholder="e.g. EMP001", layout=W)
w_join_date   = widgets.Text(value="2024-01-01", layout=W)
w_treat_date  = widgets.Text(value="2024-11-01", layout=W)
w_amount      = widgets.IntSlider(
    min=500, max=20000, step=100, value=2000,
    style={"handle_color": "#0288d1"},
    layout=widgets.Layout(width="320px"))
w_amount_lbl  = widgets.HTML()

def update_amt_lbl(change):
    w_amount_lbl.value = (
        f"<div style='font-size:13px;color:#0288d1;font-weight:700'>"
        f"₹{change['new']:,}</div>")
w_amount.observe(update_amt_lbl, names="value")
update_amt_lbl({"new": w_amount.value})

w_hospital    = widgets.Dropdown(
    options=["(Non-network)"] + policy["network_hospitals"],
    layout=W)
w_diagnosis   = widgets.Text(placeholder="e.g. Viral fever", layout=W)
w_doctor      = widgets.Text(placeholder="e.g. Dr. Sharma", layout=W)
w_doctor_reg  = widgets.Text(placeholder="e.g. KA/45678/2015", layout=W)
w_procedures  = widgets.SelectMultiple(
    options=["Root canal treatment", "Teeth whitening", "Filling", "Extraction"],
    layout=widgets.Layout(width="320px", height="82px"))
w_prev_claims = widgets.IntSlider(
    min=0, max=5, value=0,
    style={"handle_color": "#e53935"},
    layout=widgets.Layout(width="320px"))
w_has_mri     = widgets.Checkbox(value=False,
    description="Has MRI scan (>₹10,000 needs pre-auth)")
w_is_ayur     = widgets.Checkbox(value=False,
    description="Ayurvedic / Homeopathic treatment")
w_submit_btn  = widgets.Button(
    description="⚖️  Adjudicate Claim",
    button_style="primary",
    layout=widgets.Layout(width="210px", height="42px"))
w_out         = widgets.Output()

col1 = widgets.VBox([
    lbl("👤 Member Name"),      w_name,
    lbl("🆔 Member ID"),        w_member_id,
    lbl("📅 Join Date"),        w_join_date,
    lbl("📅 Treatment Date"),   w_treat_date,
    lbl("💵 Claim Amount (₹)"), w_amount, w_amount_lbl,
    lbl("🏥 Hospital"),         w_hospital,
])

col2 = widgets.VBox([
    lbl("🩺 Diagnosis"),               w_diagnosis,
    lbl("👨‍⚕️ Doctor Name"),             w_doctor,
    lbl("📋 Doctor Reg No"),           w_doctor_reg,
    lbl("🦷 Procedures (multi-select)"),w_procedures,
    lbl("🚩 Previous Claims Today"),   w_prev_claims,
    widgets.HTML("<div style='margin-top:8px'></div>"),
    w_has_mri,
    w_is_ayur,
])

submit_tab = widgets.VBox([
    widgets.HTML("<h3 style='color:#1a237e;margin:10px 0 4px'>📝 Enter Claim Details</h3>"),
    widgets.HTML("<p style='color:#888;font-size:12px;margin:0 0 10px'>"
                 "Fill in the form below and click Adjudicate.</p>"),
    widgets.HBox([col1,
                  widgets.HTML("<div style='width:32px'></div>"),
                  col2]),
    widgets.HTML("<div style='margin-top:14px'></div>"),
    w_submit_btn,
    w_out
])

# ── Tab 2 widgets ────────────────────────────────────────────────────────
w_tc_select = widgets.Dropdown(
    options=[(f"{c['case_id']} — {c['case_name']}", c["case_id"])
             for c in test_cases["test_cases"]],
    layout=widgets.Layout(width="460px"))
w_tc_btn    = widgets.Button(
    description="🔍 Run Test Case",
    button_style="info",
    layout=widgets.Layout(width="180px", height="42px"))
w_tc_out    = widgets.Output()

test_tab = widgets.VBox([
    widgets.HTML("<h3 style='color:#1a237e;margin:10px 0 4px'>🧪 Official Test Cases</h3>"),
    widgets.HTML("<p style='color:#888;font-size:12px;margin:0 0 10px'>"
                 "Select any test case and run it to see the decision vs expected output.</p>"),
    lbl("Select test case:"),
    w_tc_select,
    widgets.HTML("<div style='margin-top:12px'></div>"),
    w_tc_btn,
    w_tc_out
])

# ── Tab 3 widgets ────────────────────────────────────────────────────────
w_batch_btn = widgets.Button(
    description="▶  Run All 10 Cases",
    button_style="success",
    layout=widgets.Layout(width="200px", height="42px"))
w_batch_out = widgets.Output()

batch_tab = widgets.VBox([
    widgets.HTML("<h3 style='color:#1a237e;margin:10px 0 4px'>📊 Batch Validation</h3>"),
    widgets.HTML("<p style='color:#888;font-size:12px;margin:0 0 10px'>"
                 "Runs all 10 official test cases and validates against expected output.</p>"),
    w_batch_btn,
    w_batch_out
])

# ── Tabs ─────────────────────────────────────────────────────────────────
tabs = widgets.Tab(children=[submit_tab, test_tab, batch_tab])
tabs.set_title(0, "📝 Submit Claim")
tabs.set_title(1, "🧪 Test Cases")
tabs.set_title(2, "📊 Batch Validate")

In [10]:
# ================================
# 8. BUTTON HANDLERS
# ================================

# ── Submit Claim ─────────────────────────────────────────────────────────
def on_submit(b):
    with w_out:
        clear_output()
        display(HTML("<p style='color:#888;font-size:13px'>⏳ Processing claim...</p>"))

        selected_procedures = list(w_procedures.value)

        # ── Build bill (FIXED: teeth_whitening amount now correctly set) ──
        bill = {
            "consultation_fee": int(w_amount.value * 0.4),
            "medicines"       : int(w_amount.value * 0.4),
        }

        # If MRI selected, add mri_scan line
        if w_has_mri.value:
            bill["mri_scan"] = w_amount.value

        # If Teeth whitening selected → split bill correctly
        if "Teeth whitening" in selected_procedures:
            bill["teeth_whitening"] = int(w_amount.value * 0.35)  # 35% is cosmetic
            bill["root_canal"]      = int(w_amount.value * 0.55)  # 55% is covered
            bill.pop("medicines", None)

        # ── Build prescription ────────────────────────────────────────────
        prescription = {
            "doctor_name": w_doctor.value     or "Dr. Unknown",
            "doctor_reg" : w_doctor_reg.value or "XX/00000/2020",
            "diagnosis"  : w_diagnosis.value  or "General consultation",
            "procedures" : selected_procedures,
        }
        if w_is_ayur.value:
            prescription["treatment"] = "Panchakarma therapy"

        # ── Build full claim ──────────────────────────────────────────────
        claim = {
            "member_id"               : w_member_id.value  or "EMP000",
            "member_name"             : w_name.value        or "Member",
            "member_join_date"        : w_join_date.value,
            "treatment_date"          : w_treat_date.value,
            "claim_amount"            : w_amount.value,
            "hospital"                : None if w_hospital.value == "(Non-network)"
                                             else w_hospital.value,
            "previous_claims_same_day": w_prev_claims.value,
            "documents": {
                "prescription": prescription,
                "bill"        : bill
            }
        }

        result      = adjudicate_claim(claim, policy)
        ai_text     = get_ai_explanation(claim, result)
        clear_output()
        display(HTML(build_result_html(claim, result, ai_text)))

w_submit_btn.on_click(on_submit)


# ── Test Case Runner ──────────────────────────────────────────────────────
def on_run_tc(b):
    with w_tc_out:
        clear_output()
        display(HTML("<p style='color:#888;font-size:13px'>⏳ Running test case...</p>"))

        case_id  = w_tc_select.value
        case     = next(c for c in test_cases["test_cases"] if c["case_id"] == case_id)
        claim    = case["input_data"]
        expected = case["expected_output"]
        result   = adjudicate_claim(claim, policy)
        ai_text  = get_ai_explanation(claim, result)

        decision_ok = result["decision"] == expected["decision"]
        amount_ok   = ("approved_amount" not in expected) or \
                      (result.get("approved_amount") == expected.get("approved_amount"))
        passed      = decision_ok and amount_ok

        badge_html = (
            "<span style='background:#2e7d32;color:white;padding:4px 14px;"
            "border-radius:20px;font-size:13px;font-weight:600'>✅ PASS</span>"
            if passed else
            "<span style='background:#c62828;color:white;padding:4px 14px;"
            "border-radius:20px;font-size:13px;font-weight:600'>❌ FAIL</span>"
        )

        clear_output()
        display(HTML(
            f"<div style='font-family:Arial;padding:10px 4px;font-size:13px;"
            f"color:#555;margin-bottom:4px'>"
            f"Expected: <b>{expected['decision']}</b>"
            f"&nbsp;&nbsp;&nbsp;{badge_html}</div>"
        ))
        display(HTML(build_result_html(claim, result, ai_text)))

w_tc_btn.on_click(on_run_tc)


# ── Batch Validate ────────────────────────────────────────────────────────
def on_batch(b):
    with w_batch_out:
        clear_output()
        display(HTML("<p style='color:#888;font-size:13px'>⏳ Running all 10 cases...</p>"))

        rows   = ""
        passed = 0

        dec_colors = {
            "APPROVED"     : "#2e7d32",
            "REJECTED"     : "#c62828",
            "PARTIAL"      : "#e65100",
            "MANUAL_REVIEW": "#1565c0"
        }

        for i, case in enumerate(test_cases["test_cases"]):
            claim    = case["input_data"]
            expected = case["expected_output"]
            result   = adjudicate_claim(claim, policy)

            ok = (result["decision"] == expected["decision"]) and \
                 (("approved_amount" not in expected) or
                  result.get("approved_amount") == expected.get("approved_amount"))
            if ok:
                passed += 1

            status_cell = (
                "<td style='padding:9px 14px;color:#2e7d32;font-weight:700;"
                "font-size:15px'>✅</td>"
                if ok else
                "<td style='padding:9px 14px;color:#c62828;font-weight:700;"
                "font-size:15px'>❌</td>"
            )

            got_amt = f"₹{result['approved_amount']:,}" \
                      if result.get("approved_amount") is not None else "—"
            exp_amt = f"₹{expected['approved_amount']:,}" \
                      if expected.get("approved_amount") is not None else "—"
            dec_col = dec_colors.get(result["decision"], "#333")
            bg      = "#f9f9f9" if i % 2 == 0 else "#ffffff"

            rows += f"""
            <tr style='background:{bg}'>
              <td style='padding:9px 14px;font-weight:700;color:#555'>{case['case_id']}</td>
              <td style='padding:9px 14px;font-size:12px;color:#777'>{case['case_name']}</td>
              <td style='padding:9px 14px;color:{dec_col};font-weight:700'>
                {result['decision']}</td>
              <td style='padding:9px 14px;font-weight:600'>{got_amt}</td>
              <td style='padding:9px 14px;color:#aaa'>{exp_amt}</td>
              {status_cell}
            </tr>"""

        score_pct   = int(passed / 10 * 100)
        score_color = "#2e7d32" if passed == 10 else "#e65100"
        score_msg   = "🎉 PERFECT SCORE!" if passed == 10 else f"⚠️ {10-passed} case(s) need review"

        clear_output()
        display(HTML(f"""
        <div style="font-family:Arial,sans-serif;max-width:740px">

          <table style="width:100%;border-collapse:collapse;border-radius:10px;
                        overflow:hidden;box-shadow:0 4px 14px rgba(0,0,0,0.1)">
            <thead>
              <tr style="background:#1a237e;color:white">
                <th style="padding:11px 14px;text-align:left">ID</th>
                <th style="padding:11px 14px;text-align:left">Case Name</th>
                <th style="padding:11px 14px;text-align:left">Decision</th>
                <th style="padding:11px 14px;text-align:left">Got ₹</th>
                <th style="padding:11px 14px;text-align:left">Exp ₹</th>
                <th style="padding:11px 14px;text-align:left">Pass</th>
              </tr>
            </thead>
            <tbody>{rows}</tbody>
          </table>

          <div style="margin-top:16px;padding:16px 22px;border-radius:10px;
                      background:linear-gradient(135deg,#e8f5e9,#f1f8e9);
                      border-left:5px solid {score_color}">
            <div style="font-size:20px;font-weight:700;color:{score_color}">
              🎯 {passed}/10 passed &nbsp;—&nbsp; {score_msg}
            </div>
            <div style="margin-top:6px;background:#e0e0e0;border-radius:6px;height:10px">
              <div style="background:{score_color};width:{score_pct}%;
                          height:10px;border-radius:6px;transition:width 0.5s"></div>
            </div>
          </div>

        </div>
        """))

In [11]:
display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:720px;margin:0 auto 16px">
  <div style="background:linear-gradient(135deg,#1a237e 0%,#0288d1 100%);
              padding:22px 28px;border-radius:14px;color:white;
              box-shadow:0 4px 16px rgba(0,0,0,0.2)">
    <div style="font-size:28px;font-weight:700">🏥 Plum OPD Adjudication Tool</div>
    <div style="font-size:13px;opacity:0.85;margin-top:6px">
      ✅ Free AI &nbsp;·&nbsp; No API Key &nbsp;·&nbsp; Runs in Google Colab
    </div>
  </div>
</div>
"""))
display(tabs)
w_batch_btn.on_click(on_batch)